In [ ]:
# Cell 1: Dependencies
#!pip install google-generativeai datasets pandas pyarrow huggingface_hub langid

# Cell 2: Imports
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
import os
import json
import time
import google.generativeai as genai
from datetime import datetime
import re
import langid
from collections import Counter
import hashlib
from huggingface_hub import login, upload_file, hf_hub_download

In [ ]:
# Cell 3: Iterative Deep Thinking Configuration (EASY TO MODIFY)
DOMAIN_CONFIG = {
    'domain': 'Indonesian Legal System - Iterative Deep Thinking',
    'target_language': 'id',
    'num_variants': 2,  # Even fewer for very complex thinking
    
    'topics': [
        'Analisis Kasus Hukum Pidana Kompleks',
        'Dilema Etis dalam Hukum Perdata',
        'Interpretasi Konstitusi dan Precedent',
        'Konflik Hukum Keluarga Modern',
        'Strategi Litigasi Bisnis',
        'Reformasi Hukum Ketenagakerjaan',
        'Kebijakan Hukum Lingkungan Berkelanjutan',
        'Implementasi HAM dalam Praktik',
        'Inovasi Prosedur Pengadilan',
        'Evolusi Hukum Properti Digital'
    ]
}

# Thinking process structure components - EASILY MODIFIABLE
THINKING_STRUCTURE = {
    'initial_analysis': [
        'Analisis awal masalah',
        'Identifikasi isu-isu utama',
        'Pengumpulan fakta-fakta kunci',
        'Penentuan konteks hukum yang relevan'
    ],
    
    'deep_exploration': [
        'Penelusuran dasar hukum yang berlaku',
        'Analisis precedent dan yurisprudensi',
        'Pertimbangan berbagai perspektif stakeholder',
        'Evaluasi dampak dan konsekuensi potensial'
    ],
    
    'critical_evaluation': [
        'Pemeriksaan ulang asumsi-asumsi awal',
        'Identifikasi kelemahan dalam argumentasi',
        'Pertimbangan counter-argument',
        'Validasi terhadap sumber-sumber hukum'
    ],
    
    'iterative_refinement': [
        'Revisi pandangan berdasarkan evaluasi',
        'Integrasi insight baru yang ditemukan',
        'Penyesuaian kesimpulan sementara',
        'Konfirmasi kembali relevansi dan akurasi'
    ],
    
    'final_synthesis': [
        'Konsolidasi seluruh analisis',
        'Penarikan kesimpulan yang terintegrasi',
        'Formulasi rekomendasi yang actionable',
        'Penyusunan jawaban yang concise namun comprehensive'
    ]
}

REASONING_INDICATORS = [
    # Initial reasoning
    'berdasarkan analisis awal', 'dari pemahaman pertama', 'pada pandangan pertama',
    
    # Deep analysis
    'setelah meneliti lebih lanjut', 'dari kajian mendalam', 'berdasarkan penelusuran',
    
    # Critical evaluation
    'namun perlu dipertimbangkan', 'di sisi lain', 'akan tetapi', 'sebaliknya',
    'setelah mempertimbangkan ulang', 'dengan memeriksa kembali',
    
    # Iterative refinement
    'setelah refleksi lebih lanjut', 'dengan mempertimbangkan kembali', 'revisi pemahaman menunjukkan',
    'insight tambahan mengungkapkan', 'analisis ulang memperlihatkan',
    
    # Final synthesis
    'berdasarkan keseluruhan analisis', 'dengan mengintegrasikan semua faktor',
    'kesimpulan akhir menunjukkan', 'secara komprehensif dapat disimpulkan'
]

FACT_CHECK_PHRASES = [
    'verifikasi terhadap', 'konfirmasi bahwa', 'validasi menunjukkan',
    'cross-check dengan', 'pemeriksaan ulang mengkonfirmasi',
    'fact-checking mengungkapkan', 'validasi sumber menunjukkan'
]

# System prompt for iterative deep thinking
SYSTEM_PROMPT = """Anda adalah seorang ahli hukum Indonesia dengan kemampuan analisis yang sangat mendalam. Tugas Anda adalah melakukan proses berpikir iteratif yang sangat panjang dan menyeluruh, kemudian memberikan jawaban yang concise dan direct. Proses thinking harus menunjukkan loop iteratif, fact-checking, dan reconsideration berkali-kali sebelum mencapai kesimpulan."""

# Iterative deep thinking prompt
def create_synthesis_prompt(topic, num_variants):
    structure_example = """
1. ANALISIS AWAL: [pemahaman pertama]
2. PENELUSURAN MENDALAM: [kajian komprehensif]  
3. EVALUASI KRITIS: [pemeriksaan ulang asumsi]
4. ITERASI 1: [revisi berdasarkan temuan baru]
5. FACT-CHECK: [verifikasi ulang data dan sumber]
6. ITERASI 2: [penyempurnaan pemahaman]
7. EVALUASI ULANG: [pemeriksaan kembali dari sudut berbeda]
8. ITERASI 3: [integrasi insight final]
9. SINTESIS AKHIR: [konsolidasi seluruh analisis]"""
    
    json_format = []
    for i in range(num_variants):
        json_format.append(f'  {{"question": "pertanyaan kompleks {i+1} tentang {topic}", "thinking": "proses berpikir iteratif yang sangat panjang dengan multiple loops", "answer": "jawaban singkat dan direct {i+1}"}}')
    
    json_template = "[\n" + ",\n".join(json_format) + "\n]"
    
    return f"""Buat {num_variants} pasangan pertanyaan-jawaban dengan ITERATIVE DEEP THINKING tentang: "{topic}"

PERSYARATAN ITERATIVE THINKING:
1. Thinking process harus SANGAT PANJANG (minimal 800 kata) dengan struktur iteratif
2. Harus ada minimal 3 loop iterasi dengan reconsideration dan fact-checking
3. Tunjukkan proses revisi pemikiran dan refinement berkali-kali
4. Answer harus SINGKAT (100-200 kata) namun direct dan comprehensive
5. Gunakan bahasa Indonesia formal dengan terminologi hukum yang tepat

STRUKTUR THINKING PROCESS:
{structure_example}

INDIKATOR YANG HARUS ADA:
- Frasa iterative: "setelah mempertimbangkan ulang", "revisi pemahaman", "iterasi selanjutnya"
- Fact-checking: "verifikasi menunjukkan", "cross-check dengan", "validasi sumber"
- Multiple perspectives: "dari sudut pandang lain", "perspektif berbeda", "angle alternatif"
- Refinement: "penyempurnaan analisis", "integrasi insight baru", "konsolidasi pemahaman"

RASIO TARGET:
- Thinking: 800-1500 kata (sangat detail dengan loop iteratif)
- Answer: 100-200 kata (concise dan direct)

FORMAT JSON:
{json_template}

Berikan HANYA JSON array yang valid tanpa penjelasan tambahan."""

In [ ]:
# Cell 4: Main Configuration
CONFIG = {
    #'gemini_api_key': '',
    'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    'huggingface_token': '',
    'output_repository': 'Azzindani/ID_Legal_QA_SynDeepThink',  # Change this
    
    # Processing settings
    'chunk_size': 20,  # Questions per chunk
    'model_name': 'gemini-2.5-flash',
    'temperature': 0.7,
    'max_output_tokens': 4000,
    'base_tokens_per_variant': 3000,  # More tokens for deep thinking
    
    # Quality settings
    'min_question_length': 30,
    'min_answer_length': 100,
    'max_question_length': 2000,
    'max_answer_length': 1000,
    'target_language_confidence': 0.8,  # Minimum confidence for language detection
    
    # Generation settings
    'questions_per_topic': 20,  # How many questions to generate per topic
    'progress_file': 'synthesis_progress.json'
}

In [ ]:
# Cell 5: Authentication
genai.configure(api_key=CONFIG['gemini_api_key'])
login(token=CONFIG['huggingface_token'])

In [ ]:
# Cell 6: Language Detection Scorer
class LanguageScorer:
    def __init__(self, target_language='id'):
        self.target_language = target_language
        langid.set_languages([target_language, 'en'])  # Always include English as fallback
    
    def detect_language(self, text):
        """Detect language and return (language, confidence)"""
        try:
            lang, confidence = langid.classify(text)
            return lang, confidence
        except:
            return 'unknown', 0.0
    
    def score_language_accuracy(self, question, answer):
        """Score based on language detection accuracy"""
        q_lang, q_conf = self.detect_language(question)
        a_lang, a_conf = self.detect_language(answer)
        
        scores = {
            'question_language': q_lang,
            'question_confidence': q_conf,
            'answer_language': a_lang, 
            'answer_confidence': a_conf,
            'question_correct_language': q_lang == self.target_language,
            'answer_correct_language': a_lang == self.target_language,
            'question_meets_threshold': q_conf >= CONFIG['target_language_confidence'],
            'answer_meets_threshold': a_conf >= CONFIG['target_language_confidence'],
        }
        
        # Overall language score
        language_accuracy = 0.0
        if scores['question_correct_language'] and scores['answer_correct_language']:
            language_accuracy = (q_conf + a_conf) / 2
        elif scores['question_correct_language'] or scores['answer_correct_language']:
            language_accuracy = max(q_conf, a_conf) * 0.5  # Penalty for mixed languages
        
        scores['language_accuracy'] = language_accuracy
        scores['passes_language_check'] = (
            scores['question_correct_language'] and 
            scores['answer_correct_language'] and
            scores['question_meets_threshold'] and 
            scores['answer_meets_threshold']
        )
        
        return scores

In [ ]:
# Cell 7: Progress Manager (same as original but adapted)
class ProgressManager:
    def __init__(self):
        self.progress_data = {
            'processed_topics': {},
            'current_topic': None,
            'total_questions_generated': 0,
            'total_qa_pairs_created': 0,
            'request_count': 0,
            'start_time': None,
            'last_update': None,
            'errors': [],
            'statistics': {
                'language_accuracy': {'passed': 0, 'failed': 0},
                'avg_language_score': 0.0,
                'topic_completion': {}
            }
        }
        self.load_progress()
    
    def load_progress(self):
        """Load progress from output repository"""
        try:
            progress_path = hf_hub_download(
                repo_id=CONFIG['output_repository'],
                filename=CONFIG['progress_file'],
                repo_type="dataset"
            )
            with open(progress_path, 'r') as f:
                saved_progress = json.load(f)
                self.progress_data.update(saved_progress)
            print(f"Progress loaded: {self.progress_data['total_qa_pairs_created']} QA pairs created")
        except Exception as e:
            print(f"No existing progress found, starting fresh: {e}")
            self.progress_data['start_time'] = datetime.now().isoformat()
            # Initialize fresh progress for all topics
            for topic in DOMAIN_CONFIG['topics']:
                self.progress_data['processed_topics'][topic] = 0
    
    def save_progress(self):
        try:
            self.progress_data['last_update'] = datetime.now().isoformat()
            
            def convert_types(obj):
                if isinstance(obj, dict):
                    return {k: convert_types(v) for k, v in obj.items()}
                elif isinstance(obj, list):
                    return [convert_types(v) for v in obj]
                elif hasattr(obj, 'item'):
                    return obj.item()
                elif hasattr(obj, 'tolist'):
                    return obj.tolist()
                else:
                    return obj
            
            clean_data = convert_types(self.progress_data)
            local_path = f"./{CONFIG['progress_file']}"
            
            with open(local_path, 'w') as f:
                json.dump(clean_data, f, indent=2)
            
            upload_file(
                path_or_fileobj=local_path,
                path_in_repo=CONFIG['progress_file'],
                repo_id=CONFIG['output_repository'],
                repo_type="dataset",
                commit_message=f"Progress update: {self.progress_data['total_qa_pairs_created']} QA pairs created"
            )
            print(f"Progress saved to repository")
        except Exception as e:
            print(f"Failed to save progress: {e}")
    
    def mark_topic_progress(self, topic, questions_done, total_questions):
        if topic not in self.progress_data['processed_topics']:
            self.progress_data['processed_topics'][topic] = 0
        self.progress_data['processed_topics'][topic] = questions_done
        self.progress_data['statistics']['topic_completion'][topic] = {
            'done': questions_done,
            'total': total_questions,
            'percentage': (questions_done / total_questions) * 100 if total_questions > 0 else 0
        }
    
    def add_qa_pairs(self, count, language_stats=None):
        self.progress_data['total_qa_pairs_created'] += count
        if language_stats:
            stats = self.progress_data['statistics']
            if language_stats['passes_language_check']:
                stats['language_accuracy']['passed'] += count
            else:
                stats['language_accuracy']['failed'] += count
    
    def show_progress(self):
        total_expected = len(DOMAIN_CONFIG['topics']) * CONFIG['questions_per_topic']
        completed = self.progress_data['total_qa_pairs_created']
        percentage = (completed / total_expected) * 100 if total_expected > 0 else 0
        
        print(f"Overall Progress: {completed:,}/{total_expected:,} ({percentage:.1f}%)")
        print(f"Topics completed: {len([t for t, p in self.progress_data['processed_topics'].items() if p >= CONFIG['questions_per_topic']])}/{len(DOMAIN_CONFIG['topics'])}")
        
        # Show topic-wise progress
        for topic in DOMAIN_CONFIG['topics']:
            done = self.progress_data['processed_topics'].get(topic, 0)
            total = CONFIG['questions_per_topic']
            pct = (done / total) * 100 if total > 0 else 0
            status = "✓" if done >= total else "→"
            print(f"  {status} {topic}: {done}/{total} ({pct:.1f}%)")

In [ ]:
# Cell 8: Knowledge Distillation Synthesizer
class KnowledgeDistillationSynthesizer:
    def __init__(self, progress_manager):
        # Calculate max tokens dynamically
        
        self.model = genai.GenerativeModel(
            CONFIG['model_name'],
            generation_config=genai.types.GenerationConfig(
                temperature=CONFIG['temperature'],
                max_output_tokens=DOMAIN_CONFIG['num_variants'] * CONFIG['max_output_tokens'],
                top_p=0.8,
                top_k=40
            ),
            system_instruction=SYSTEM_PROMPT  # Now uses the external prompt
        )
        self.language_scorer = LanguageScorer(DOMAIN_CONFIG['target_language'])
        self.progress_manager = progress_manager
        self.request_count = progress_manager.progress_data['request_count']
    
    def generate_questions_for_topic(self, topic):
        """Generate questions for a specific topic"""
        prompt = DOMAIN_CONFIG['question_prompt'].format(
            num_variants=DOMAIN_CONFIG['num_variants'],
            topic=topic
        )
        
        try:
            time.sleep(2)  # Rate limiting
            response = self.model.generate_content(prompt)
            self.request_count += 1
            self.progress_manager.progress_data['request_count'] = self.request_count
            
            if not response.candidates or not response.candidates[0].content:
                return None
            
            response_text = response.candidates[0].content.parts[0].text.strip()
            
            # Extract JSON
            json_content = None
            if "```json" in response_text:
                json_content = response_text.split("```json")[1].split("```")[0].strip()
            elif "```" in response_text:
                json_content = response_text.split("```")[1].strip()
            else:
                json_match = re.search(r'\[.*?\]', response_text, re.DOTALL)
                if json_match:
                    json_content = json_match.group(0)
            
            if not json_content:
                return None
            
            try:
                questions = json.loads(json_content)
                if isinstance(questions, list):
                    return [q.strip() for q in questions if isinstance(q, str) and len(q.strip()) > 10]
            except:
                pass
            
            return None
            
        except Exception as e:
            print(f"Error generating questions for {topic}: {e}")
            return None
    
    def generate_answer_for_question(self, question):
        """Generate answer for a specific question"""
        prompt = DOMAIN_CONFIG['answer_prompt'].format(question=question)
        
        try:
            time.sleep(2)  # Rate limiting
            response = self.model.generate_content(prompt)
            self.request_count += 1
            self.progress_manager.progress_data['request_count'] = self.request_count
            
            if not response.candidates:
                print(f"    No candidates returned for answer generation")
                return None
                
            candidate = response.candidates[0]
            
            # Check finish reason like in your original code
            if candidate.finish_reason == 2:  # SAFETY
                print(f"    Answer filtered by safety")
                return None
            elif candidate.finish_reason == 3:  # RECITATION
                print(f"    Answer flagged as recitation")
                return None
            elif candidate.finish_reason == 4:  # OTHER
                print(f"    Answer generation failed - other reason")
                return None
            
            if not candidate.content or not candidate.content.parts:
                print(f"    No content in answer response")
                return None
            
            if len(candidate.content.parts) == 0:
                print(f"    Empty parts in answer response")
                return None
                
            answer = candidate.content.parts[0].text.strip()
            return answer if len(answer) >= CONFIG['min_answer_length'] else None
            
        except Exception as e:
            print(f"    Error generating answer: {e}")
            return None
    
    def synthesize_topic(self, topic):
        """Generate iterative deep thinking QA pairs - SAVE EVERYTHING with stats"""
        print(f"\nSynthesizing ITERATIVE DEEP THINKING topic: {topic}")
        
        already_done = self.progress_manager.progress_data['processed_topics'].get(topic, 0)
        target_count = CONFIG['questions_per_topic']
        
        if already_done >= target_count:
            print(f"Topic {topic} already completed ({already_done}/{target_count})")
            return []
        
        results = []
        questions_needed = target_count - already_done
        
        print(f"Need {questions_needed} more ITERATIVE THINKING QA pairs")
        
        # Statistics tracking
        stats = {
            'total_generated': 0,
            'safety_filtered': 0,
            'quality_grades': {'Excellent': 0, 'Good': 0, 'Fair': 0, 'Poor': 0},
            'short_thinking': 0,
            'long_answers': 0,
            'accepted': 0
        }
        
        while len(results) < questions_needed:
            variants = self.generate_qa_batch(topic)
            if not variants:
                print(f"Failed to generate batch for {topic}")
                break
            
            stats['total_generated'] += len(variants)
            print(f"Generated {len(variants)} iterative thinking pairs, processing ALL...")
            
            for variant in variants:
                if len(results) >= questions_needed:
                    break
                
                question = variant.get('question', '').strip()
                thinking = variant.get('thinking', '').strip()
                answer = variant.get('answer', '').strip()
                
                # Basic validation only - accept almost everything
                if len(question) < 10 or len(thinking) < 20 or len(answer) < 10:
                    continue
                
                thinking_words = len(thinking.split())
                answer_words = len(answer.split())
                
                # Track statistics but don't reject
                if thinking_words < 500:
                    stats['short_thinking'] += 1
                if answer_words > 300:
                    stats['long_answers'] += 1
                
                # Assess quality for statistics
                thinking_quality = self.assess_thinking_quality(thinking)
                stats['quality_grades'][thinking_quality['thinking_quality_grade']] += 1
                
                # Score language
                language_scores = self.language_scorer.score_language_accuracy(question, f"{thinking} {answer}")
                
                result = {
                    'topic': topic,
                    'question': question,
                    'thinking': thinking,
                    'answer': answer,
                    'question_length': len(question),
                    'thinking_length': len(thinking),
                    'answer_length': len(answer),
                    'question_word_count': len(question.split()),
                    'thinking_word_count': thinking_words,
                    'answer_word_count': answer_words,
                    'thinking_to_answer_ratio': thinking_words / answer_words if answer_words > 0 else 0,
                    'meets_thinking_target': thinking_words >= 500,
                    'meets_answer_target': answer_words <= 200,
                    'optimal_structure': thinking_words >= 500 and answer_words <= 200,
                    'timestamp': datetime.now().isoformat(),
                    **thinking_quality,
                    **language_scores
                }
                
                results.append(result)
                stats['accepted'] += 1
                
                # Update progress
                current_total = already_done + len(results)
                self.progress_manager.mark_topic_progress(topic, current_total, target_count)
                self.progress_manager.add_qa_pairs(1, language_scores)
                
                # Show detailed statistics
                optimal_mark = "✓" if result['optimal_structure'] else "○"
                print(f"  {optimal_mark} QA {len(results)}/{questions_needed} - Quality: {thinking_quality['thinking_quality_grade']} - T/A: {result['thinking_to_answer_ratio']:.1f} - Loops: {thinking_quality['iteration_loops']} - Facts: {thinking_quality['fact_checks']}")
            
            # Show batch statistics
            print(f"  Batch Stats - Generated: {len(variants)}, Accepted: {stats['accepted']}, Quality: {dict(stats['quality_grades'])}")
            
            if len(results) > 0:
                self.progress_manager.save_progress()
        
        # Final topic statistics
        if results:
            optimal_count = sum(1 for r in results if r['optimal_structure'])
            avg_thinking_words = sum(r['thinking_word_count'] for r in results) / len(results)
            avg_answer_words = sum(r['answer_word_count'] for r in results) / len(results)
            avg_ratio = sum(r['thinking_to_answer_ratio'] for r in results) / len(results)
            
            print(f"\nTOPIC SUMMARY - {topic}:")
            print(f"  Total saved: {len(results)} QA pairs")
            print(f"  Optimal structure: {optimal_count}/{len(results)} ({(optimal_count/len(results)*100):.1f}%)")
            print(f"  Avg thinking words: {avg_thinking_words:.0f}")
            print(f"  Avg answer words: {avg_answer_words:.0f}")
            print(f"  Avg T/A ratio: {avg_ratio:.1f}")
            print(f"  Quality distribution: {dict(stats['quality_grades'])}")
        
        print(f"Completed {topic}: {len(results)} QA pairs saved with full statistics")
        return results
    
    def assess_thinking_quality(self, thinking_text):
        """Assess the quality of iterative thinking process"""
        thinking_lower = thinking_text.lower()
        
        # Count iterative indicators
        iterative_count = sum(1 for phrase in REASONING_INDICATORS if phrase in thinking_lower)
        fact_check_count = sum(1 for phrase in FACT_CHECK_PHRASES if phrase in thinking_lower)
        
        # Check for thinking structure components
        structure_coverage = 0
        for phase, indicators in THINKING_STRUCTURE.items():
            phase_found = any(indicator.lower() in thinking_lower for indicator in indicators)
            if phase_found:
                structure_coverage += 1
        
        # Count revision/iteration indicators
        iteration_phrases = [
            'iterasi', 'revisi', 'pertimbangan ulang', 'pemeriksaan kembali',
            'setelah mempertimbangkan', 'refleksi lebih lanjut', 'analisis ulang'
        ]
        iteration_count = sum(1 for phrase in iteration_phrases if phrase in thinking_lower)
        
        # Count perspective shifts
        perspective_phrases = [
            'di sisi lain', 'sudut pandang', 'perspektif', 'sebaliknya',
            'dari angle berbeda', 'dengan mempertimbangkan', 'namun'
        ]
        perspective_count = sum(1 for phrase in perspective_phrases if phrase in thinking_lower)
        
        return {
            'iterative_indicators': iterative_count,
            'fact_checks': fact_check_count,
            'structure_coverage': structure_coverage,
            'iteration_loops': iteration_count,
            'perspective_shifts': perspective_count,
            'total_depth_score': iterative_count + fact_check_count + iteration_count + perspective_count,
            'has_deep_iteration': iteration_count >= 3,
            'has_fact_checking': fact_check_count >= 2,
            'has_multiple_perspectives': perspective_count >= 3,
            'thinking_quality_grade': self.grade_thinking_quality(
                iterative_count, fact_check_count, iteration_count, perspective_count
            )
        }
    
    def grade_thinking_quality(self, iterative, fact_check, iteration, perspective):
        """Grade the overall thinking quality"""
        total_score = iterative + fact_check + iteration + perspective
        
        if total_score >= 15 and iteration >= 3 and fact_check >= 2:
            return 'Excellent'
        elif total_score >= 10 and iteration >= 2:
            return 'Good'
        elif total_score >= 6:
            return 'Fair'
        else:
            return 'Poor'
    
    def generate_qa_batch(self, topic):
        """Generate multiple QA pairs - handle safety filters gracefully"""
        prompt = create_synthesis_prompt(topic, DOMAIN_CONFIG['num_variants'])
        
        try:
            time.sleep(4)
            response = self.model.generate_content(prompt)
            self.request_count += 1
            self.progress_manager.progress_data['request_count'] = self.request_count
            
            if not response.candidates:
                print(f"    No candidates returned (likely safety filtered)")
                return None
            
            candidate = response.candidates[0]
            
            # Handle safety filters but don't immediately fail
            if candidate.finish_reason == 2:  # SAFETY
                print(f"    Content filtered by safety - trying alternative approach")
                # Try with a more neutral prompt
                neutral_prompt = f"""Buat {DOMAIN_CONFIG['num_variants']} contoh kasus akademis tentang {topic} dalam konteks hukum Indonesia. Gunakan pendekatan teoretis dan akademis.
    
    FORMAT JSON:
    [
      {{"question": "pertanyaan akademis 1", "thinking": "analisis akademis panjang", "answer": "kesimpulan singkat"}}
    ]"""
                
                try:
                    time.sleep(2)
                    response = self.model.generate_content(neutral_prompt)
                    if response.candidates:
                        candidate = response.candidates[0]
                        if candidate.finish_reason != 2:  # Not safety filtered
                            print(f"    Alternative approach successful")
                        else:
                            print(f"    Alternative also filtered - skipping this batch")
                            return None
                    else:
                        return None
                except:
                    return None
            elif candidate.finish_reason == 3:
                print(f"    Content flagged as recitation")
                return None
            elif candidate.finish_reason == 4:
                print(f"    API error occurred")
                return None
            
            if not candidate.content or not candidate.content.parts:
                return None
            
            response_text = candidate.content.parts[0].text.strip()
            
            # Same JSON extraction as before
            json_content = None
            if "```json" in response_text:
                json_content = response_text.split("```json")[1].split("```")[0].strip()
            elif "```" in response_text:
                json_content = response_text.split("```")[1].strip()
            else:
                json_match = re.search(r'\[.*?\]', response_text, re.DOTALL)
                if json_match:
                    json_content = json_match.group(0)
            
            if not json_content:
                return None
            
            json_content = json_content.replace('\n', ' ')
            json_content = re.sub(r'\s+', ' ', json_content)
            json_content = re.sub(r',\s*}', '}', json_content)
            json_content = re.sub(r',\s*]', ']', json_content)
            
            try:
                variants = json.loads(json_content)
                if isinstance(variants, list):
                    return [v for v in variants if isinstance(v, dict) and 'question' in v and 'answer' in v]
            except:
                pass
            
            return None
            
        except Exception as e:
            print(f"Error generating batch: {e}")
            return None

In [ ]:
# Cell 9: Main Processing Functions
def process_topics(topics_to_process=None):
    """Process specific topics or all topics"""
    if topics_to_process is None:
        topics_to_process = DOMAIN_CONFIG['topics']
    
    synthesizer = KnowledgeDistillationSynthesizer(progress_manager)
    all_results = []
    
    for topic in topics_to_process:
        try:
            results = synthesizer.synthesize_topic(topic)
            all_results.extend(results)
            
            # Save chunk after each topic
            if results:
                save_topic_chunk(topic, results)
            
        except Exception as e:
            print(f"Error processing topic {topic}: {e}")
            continue
    
    return all_results

def save_topic_chunk(topic, results):
    """Save results for a topic as a chunk"""
    if not results:
        return
    
    try:
        df = pd.DataFrame(results)
        
        # Clean topic name for filename
        clean_topic = re.sub(r'[^\w\s-]', '', topic).strip().replace(' ', '_')
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"topic_{clean_topic}_{timestamp}.parquet"
        
        temp_filepath = f"/tmp/{filename}"
        df.to_parquet(temp_filepath, index=False)
        
        upload_file(
            path_or_fileobj=temp_filepath,
            path_in_repo=filename,
            repo_id=CONFIG['output_repository'],
            repo_type="dataset",
            commit_message=f"Topic synthesis: {topic} - {len(results)} QA pairs"
        )
        
        print(f"Uploaded {filename} ({len(results)} QA pairs)")
        
        # Clean up temp file
        try:
            os.remove(temp_filepath)
        except:
            pass
            
    except Exception as e:
        print(f"Failed to save topic chunk {topic}: {e}")

def continue_synthesis():
    """Continue synthesis from where we left off"""
    progress_manager.show_progress()
    
    # Find topics that need more work
    incomplete_topics = []
    for topic in DOMAIN_CONFIG['topics']:
        done = progress_manager.progress_data['processed_topics'].get(topic, 0)
        if done < CONFIG['questions_per_topic']:
            incomplete_topics.append(topic)
    
    if incomplete_topics:
        print(f"\nContinuing synthesis for {len(incomplete_topics)} incomplete topics...")
        process_topics(incomplete_topics)
    else:
        print("All topics completed!")

def analyze_deep_thinking_quality():
    """Analyze the quality of deep thinking content"""
    # This would analyze uploaded files for thinking quality metrics
    print("Deep Thinking Quality Analysis:")
    print("- Average thinking steps per response")
    print("- Reasoning indicators usage")
    print("- Analysis depth metrics")
    print("- Multi-perspective coverage")

In [ ]:
# Cell 10: Utility Functions
def update_domain_config(domain=None, language=None, topics=None):
    """Update domain configuration"""
    if domain:
        DOMAIN_CONFIG['domain'] = domain
    if language:
        DOMAIN_CONFIG['target_language'] = language
    if topics:
        DOMAIN_CONFIG['topics'] = topics
    
    print(f"Updated domain config:")
    print(f"  Domain: {DOMAIN_CONFIG['domain']}")
    print(f"  Language: {DOMAIN_CONFIG['target_language']}")
    print(f"  Topics: {len(DOMAIN_CONFIG['topics'])}")

def analyze_language_quality():
    """Analyze language detection quality"""
    stats = progress_manager.progress_data.get('statistics', {})
    lang_accuracy = stats.get('language_accuracy', {'passed': 0, 'failed': 0})
    
    total = lang_accuracy['passed'] + lang_accuracy['failed']
    if total == 0:
        print("No language statistics available yet")
        return
    
    pass_rate = (lang_accuracy['passed'] / total) * 100
    print(f"Language Accuracy Analysis:")
    print(f"  Passed: {lang_accuracy['passed']:,} ({pass_rate:.1f}%)")
    print(f"  Failed: {lang_accuracy['failed']:,} ({100-pass_rate:.1f}%)")
    print(f"  Total: {total:,}")

def show_config():
    """Display current configuration"""
    print("Current Configuration:")
    print(f"  Domain: {DOMAIN_CONFIG['domain']}")
    print(f"  Target Language: {DOMAIN_CONFIG['target_language']}")
    print(f"  Questions per topic: {CONFIG['questions_per_topic']}")
    print(f"  Variants per request: {DOMAIN_CONFIG['num_variants']}")
    print(f"  Total expected QA pairs: {len(DOMAIN_CONFIG['topics']) * CONFIG['questions_per_topic']:,}")
    print(f"  Topics ({len(DOMAIN_CONFIG['topics'])}):")
    for i, topic in enumerate(DOMAIN_CONFIG['topics'], 1):
        print(f"    {i}. {topic}")

In [ ]:
# Initialize progress manager
try:
    progress_manager = ProgressManager()
    print("Progress manager initialized successfully")
except Exception as e:
    print(f"Warning: Could not initialize progress manager: {e}")
    progress_manager = ProgressManager()
    progress_manager.progress_data['start_time'] = datetime.now().isoformat()
    for topic in DOMAIN_CONFIG['topics']:
        progress_manager.progress_data['processed_topics'][topic] = 0

print("Knowledge Distillation QA Synthesis Pipeline Ready!")
print("=" * 50)
print("Key Functions:")
print("- show_config() - Display current configuration")
print("- continue_synthesis() - Start/continue synthesis")
print("- process_topics(['topic1', 'topic2']) - Process specific topics")
print("- progress_manager.show_progress() - Check progress") 
print("- analyze_language_quality() - Check language detection stats")
print("- update_domain_config(domain, language, topics) - Update configuration")

show_config()

# Auto-continue processing like the original code
print("\nChecking progress and continuing...")
try:
    continue_synthesis()
except Exception as e:
    print(f"Auto-start encountered an issue: {e}")
    print("Run continue_synthesis() manually if needed")